# Neural Networks with PyTorch

## Lecture 3: Introduction to Deep Learning

---

**In this lecture:**
- From Logistic Regression to Neural Networks
- Building MLPs with PyTorch
- Training on MNIST Digit Recognition
- Understanding Model Architecture

---

*Powered by PyTorch & RISE*

## Introduction: Why Neural Networks?

---

**Logistic Regression Limitations:**
- Can only learn **linear decision boundaries**
- Limited representational power for complex patterns

**Neural Networks Advantages:**
- Can learn **non-linear, complex decision boundaries**
- Universal function approximators
- Multiple layers enable hierarchical feature learning

---

**Today's Goal:** Build a Multi-Layer Perceptron (MLP) to recognize handwritten digits!

## Importing Libraries

---

We'll use **PyTorch** - the most popular deep learning framework:
- Dynamic computation graphs
- Pythonic and intuitive
- Strong GPU acceleration support

In [ ]:
# Essential libraries
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")

## The MNIST Dataset

---

**What is MNIST?**
- 60,000 training images, 10,000 test images
- 28×28 grayscale images of handwritten digits (0-9)
- Each image has a label (0, 1, 2, ..., 9)

**Why MNIST?**
- The "Hello World" of deep learning
- Simple enough to learn quickly
- Complex enough to demonstrate neural network power

---

*We'll use local CSV files to avoid network issues*

## Loading the Dataset

---

Loading MNIST from local CSV files:

In [ ]:
# Load MNIST from CSV files
print("Loading training data...")
train_df = pd.read_csv('mnist_train.csv')
print(f"Training samples: {len(train_df)}")

print("\nLoading test data...")
test_df = pd.read_csv('mnist_test.csv')
print(f"Test samples: {len(test_df)}")

# Separate features and labels
x_train = train_df.iloc[:, 1:].values  # All columns except first (label)
y_train = train_df.iloc[:, 0].values   # First column is label
x_test = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values

print(f"\nTraining data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")

## Exploring the Data

---

Let's check the class distribution:

In [ ]:
# Count samples per class
print("Training set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count} samples")

print("\nTest set class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count} samples")

## Visualizing Sample Digits

---

Let's see what the data looks like!

In [ ]:
# Display 25 random sample images
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
axes = axes.flatten()

for i, ax in enumerate(axes):
    idx = np.random.randint(0, len(x_train))
    img = x_train[idx].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {y_train[idx]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Data Preprocessing

---

**Why Preprocess?**
1. **Reshape**: Flatten 28×28 images → 784-dimensional vectors
2. **Normalize**: Scale pixel values from [0, 255] → [0, 1]
   - Faster convergence
   - More stable training
3. **Convert to Tensors**: PyTorch works with tensors, not numpy arrays

In [ ]:
# Reshape: (N, 28, 28) → (N, 784)
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

print(f"Flattened shape: {x_train.shape}")

# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f"Pixel value range: [{x_train.min():.2f}, {x_train.max():.2f}]")

## Converting to PyTorch Tensors

---

PyTorch requires:
- **Input features**: FloatTensor
- **Labels**: LongTensor (for classification)

In [ ]:
# Convert numpy arrays to PyTorch tensors
X_train = torch.FloatTensor(x_train)
y_train = torch.LongTensor(y_train)
X_test = torch.FloatTensor(x_test)
y_test = torch.LongTensor(y_test)

# Create DataLoader for mini-batch training
batch_size = 128
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

print(f"Training batches: {len(train_loader)}")
print(f"Batch size: {batch_size}")

## Building the Neural Network

---

**Our Architecture:**
```
Input (784) → Dense(256) → ReLU → Dropout → Dense(256) → ReLU → Dropout → Output(10)
```

**Key Components:**
- **Dense (Linear)**: Fully connected layer
- **ReLU**: Activation function (introduces non-linearity)
- **Dropout**: Regularization (prevents overfitting)

In [ ]:
# Define the Neural Network
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=256, num_classes=10, dropout=0.45):
        super(MLP, self).__init__()
        
        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Second hidden layer
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Output layer
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Create model instance
model = MLP(input_size=784, hidden_size=256, num_classes=10, dropout=0.45)
print("Model created successfully!")

## Model Architecture Summary

---

Let's examine our model structure and parameter count:

In [ ]:
print(model)
print("\n" + "="*50)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Calculate parameter breakdown
print("\nParameter breakdown:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.numel():,}")

## Loss Function & Optimizer

---

**Loss Function: CrossEntropyLoss**
- Combines LogSoftmax and NLLLoss
- Perfect for multi-class classification
- Measures difference between predicted and true distribution

**Optimizer: Adam**
- Adaptive learning rate
- Combines momentum and RMSProp benefits
- Default choice for most deep learning tasks

In [ ]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function: {criterion.__class__.__name__}")
print(f"Optimizer: {optimizer.__class__.__name__}")
print(f"Learning rate: {optimizer.defaults['lr']}")

## Training the Model

---

**Training Loop Steps:**
1. Forward pass: Compute predictions
2. Calculate loss: Compare with true labels
3. Backward pass: Compute gradients
4. Update weights: Optimizer step
5. Zero gradients: Prepare for next iteration

In [ ]:
# Training configuration
num_epochs = 20
print(f"Starting training for {num_epochs} epochs...")
print("="*60)

# Training history
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    
    # Epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    
    print(f"Epoch [{epoch+1:2d}/{num_epochs}] | "
          f"Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("="*60)
print("Training completed!")

## Training Progress Visualization

---

Let's plot the training loss and accuracy over epochs:

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(range(1, num_epochs+1), train_losses, 'b-', linewidth=2, marker='o')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=14)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(range(1, num_epochs+1), train_accuracies, 'g-', linewidth=2, marker='s')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training Accuracy', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best Training Accuracy: {max(train_accuracies):.2f}%")

## Evaluating on Test Set

---

Now let's see how well our model generalizes to unseen data!

In [ ]:
# Evaluation mode
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs.data, 1)
    
    total = y_test.size(0)
    correct = (predicted == y_test).sum().item()
    test_accuracy = 100 * correct / total
    
    # Calculate loss on test set
    test_loss = criterion(outputs, y_test).item()

print("="*60)
print("TEST SET EVALUATION")
print("="*60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Correct predictions: {correct}/{total}")
print("="*60)

## Confusion Matrix

---

Let's analyze which digits are most commonly confused:

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Generate predictions
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs.data, 1)
    predictions = predicted.numpy()

# Confusion matrix
cm = confusion_matrix(y_test, predictions)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, predictions, digits=4))

## Analyzing Misclassifications

---

Let's look at some examples where the model made mistakes:

In [ ]:
# Find misclassified examples
errors = (predictions != y_test)
error_indices = np.where(errors)[0]

if len(error_indices) > 0:
    # Display 10 random errors
    fig, axes = plt.subplots(2, 5, figsize=(15, 4))
    axes = axes.flatten()
    
    for i, ax in enumerate(axes):
        if i < len(error_indices):
            idx = error_indices[i]
            img = x_test[idx].reshape(28, 28)
            ax.imshow(img, cmap='gray')
            ax.set_title(f'True: {y_test[idx]} | Pred: {predictions[idx]}', 
                        fontsize=10, color='red')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f"Total misclassifications: {len(error_indices)} ({100-len(error_indices)/len(y_test)*100:.2f}% accuracy)")
else:
    print("Perfect accuracy! No errors to analyze.")

## Summary & Key Takeaways

---

**What We Learned:**

1. **Neural Networks** can learn complex, non-linear patterns
2. **PyTorch** provides an intuitive framework for building models
3. **MLP Architecture**: Input → Hidden Layers (ReLU + Dropout) → Output
4. **Training Process**: Forward → Loss → Backward → Update
5. **Best Practices**:
   - Normalize input data
   - Use dropout for regularization
   - Monitor both training and test performance

---

**Next Steps:**
- Try different architectures (more layers, different sizes)
- Experiment with hyperparameters (learning rate, batch size)
- Explore Convolutional Neural Networks (CNNs) for images

## Practice Exercises

---

**Try These:**

1. **Change Architecture**: Add a third hidden layer. Does accuracy improve?

2. **Tune Hyperparameters**: 
   - Try different learning rates (0.01, 0.0001)
   - Adjust dropout rate (0.3, 0.5, 0.6)

3. **Compare Optimizers**: Replace Adam with SGD or RMSprop

4. **Data Augmentation**: Can you artificially expand the training set?

---

*Experiment and see how each change affects performance!*

## Acknowledgments

---

**Resources:**
- [PyTorch Documentation](https://pytorch.org/docs/)
- [MNIST Dataset](http://yann.lecun.com/exdb/mnist/)
- [RISE - Jupyter Slideshow Extension](https://github.com/damianavila/RISE)

**Original Inspiration:**
- Based on lecture materials by PRASHANT BANERJEE
- Adapted from TensorFlow to PyTorch for educational purposes

---

*Happy Learning! 🚀*